# 13 — Medical Supervised Fine-Tuning (SFT)

This notebook fine-tunes the DAPT model on **instruction-formatted** medical question-answer pairs.

**Datasets:**
- `medalpaca/medical_meadow_medqa` — USMLE-style MCQ (first `cfg.MED_MEDQA_TRAIN_SIZE` examples)
- `pubmed_qa` labeled — biomedical evidence QA (first `cfg.MED_PUBMEDQA_TRAIN_SIZE` examples)

**Pipeline:**
1. Understand why instruction formatting and response masking matter
2. Load and format both datasets
3. Define a mixed batch sampler (80% MedQA, 20% PubMedQA)
4. Load the DAPT checkpoint
5. Run 5k-step SFT training loop
6. Compare DAPT vs. SFT generations on the same prompts

## 2 — Why Instruction Formatting Matters

### Prompt/Response Masking

During DAPT, the model learns to predict the *next token* for every position in the sequence.
That works well for a language-modelling objective, but for instruction following we only care
whether the model produces the **correct response** — not whether it can reproduce the question.

**Response masking** (also called "prompt masking") sets the loss to zero for all tokens that
belong to the prompt (question + options). The model only receives gradient signal from
the response tokens (the correct answer letter and explanation).

**Why not mask?**  
If we compute loss on question tokens as well, the model is incentivised to allocate capacity
toward predicting question boilerplate ("A patient presents with...") rather than the answer.
With masking, 100% of the gradient goes toward learning to answer correctly.

**The `collate_sft_batch` function** in `datasets/medical.py` handles this:
it pads sequences to equal length and returns a `labels` tensor where prompt token positions
are set to `-100` (the PyTorch cross-entropy ignore index).

## 1 — Setup

In [ ]:
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.abspath(".."))

import config as cfg
cfg.make_dirs()
cfg.print_config()

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3 — Load & Format Datasets

We load the first `cfg.MED_MEDQA_TRAIN_SIZE` examples from MedQA and
`cfg.MED_PUBMEDQA_TRAIN_SIZE` from PubMedQA. These sizes are set in `config.py` to keep
training within 5k steps at `cfg.MED_SFT_BATCH_SIZE` per step.

Both are wrapped in PyTorch Dataset classes (`MedQADataset`, `PubMedQADataset`) from
`datasets/medical.py`. The Dataset classes:
1. Call `format_*_instruction()` to produce a prompt and response string
2. Tokenize both with the medical tokenizer
3. Concatenate them with a separator token
4. Return the full token sequence and the prompt length (used for masking)

In [ ]:
from datasets import load_dataset
from loaders.medical import MedQADataset, PubMedQADataset
from tokenizer.preprocess import load_tokenizer
import config as cfg

# Load the medical tokenizer (trained in notebook 11)
med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)
pad_id  = med_tok.token_to_id("<pad>") or 0
eos_id  = med_tok.token_to_id("</s>") or 1

print(f"Tokenizer vocab size: {med_tok.get_vocab_size():,}")
print(f"  pad_id={pad_id}  eos_id={eos_id}")

In [ ]:
# ── MedQA ────────────────────────────────────────────────────────────────
raw_medqa = load_dataset(cfg.MED_MEDQA_DATASET, split="train")
# Take only the first cfg.MED_MEDQA_TRAIN_SIZE examples for SFT training
medqa_examples = list(raw_medqa.select(range(min(cfg.MED_MEDQA_TRAIN_SIZE, len(raw_medqa)))))

medqa_dataset = MedQADataset(medqa_examples, med_tok, max_len=512)
print(f"MedQA dataset   : {len(medqa_dataset):,} examples")

In [ ]:
# ── PubMedQA ─────────────────────────────────────────────────────────────
raw_pubmedqa = load_dataset(cfg.MED_PUBMEDQA_DATASET, "pqa_labeled", split="train")
pubmedqa_examples = list(raw_pubmedqa.select(range(min(cfg.MED_PUBMEDQA_TRAIN_SIZE, len(raw_pubmedqa)))))

pubmedqa_dataset = PubMedQADataset(pubmedqa_examples, med_tok, max_len=512)
print(f"PubMedQA dataset: {len(pubmedqa_dataset):,} examples")

In [ ]:
# Inspect one formatted example to verify masking is set up correctly
sample = medqa_dataset[0]
print("Keys in dataset item:", list(sample.keys()) if hasattr(sample, 'keys') else type(sample))
if isinstance(sample, dict):
    for k, v in sample.items():
        if hasattr(v, 'shape'):
            print(f"  {k}: shape={v.shape} dtype={v.dtype}")
        else:
            print(f"  {k}: {v}")

## 4 — Mixed Batch Sampler

We train on a blend of MedQA and PubMedQA: **80% MedQA, 20% PubMedQA**.

Why mix?
- MedQA alone could overfit to the USMLE format (4-option MCQ with clinical vignettes)
- PubMedQA adds yes/no/maybe decisions grounded in evidence, teaching a different reasoning mode
- The 80/20 split favours MedQA because it is directly aligned with our evaluation benchmark

The mix ratio is controlled by `cfg.MED_SFT_MIX_RATIO` (defined in config.py).
At each step we roll a uniform random number: if < MIX_RATIO, draw from MedQA; else PubMedQA.

In [ ]:
import random
import torch
from torch.utils.data import DataLoader
from loaders.medical import collate_sft_batch
import config as cfg

# Retrieve mix ratio — default to 0.8 if not in config
MIX_RATIO = getattr(cfg, "MED_SFT_MIX_RATIO", 0.8)

# Infinite data loaders — we call next() at each training step
def make_loader(dataset, batch_size, pad_id):
    """Create a DataLoader that shuffles and collates SFT batches.

    collate_sft_batch pads variable-length sequences to the longest item in
    the batch and sets prompt token labels to -100 (ignored in loss).

    Args:
        dataset   : MedQADataset or PubMedQADataset
        batch_size: number of examples per batch
        pad_id    : token ID used for padding

    Returns:
        DataLoader (shuffle=True, drop_last=True)
    """
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        collate_fn=lambda batch: collate_sft_batch(batch, pad_id),
    )

medqa_loader   = iter(make_loader(medqa_dataset,   cfg.MED_SFT_BATCH_SIZE, pad_id))
pubmedqa_loader = iter(make_loader(pubmedqa_dataset, cfg.MED_SFT_BATCH_SIZE, pad_id))

def get_sft_batch():
    """Return one batch from the mixed SFT dataset.

    With probability MIX_RATIO draw from MedQA, otherwise from PubMedQA.
    Automatically refreshes the exhausted iterator.

    Returns:
        dict with keys 'input_ids', 'labels', 'attention_mask' on device
    """
    global medqa_loader, pubmedqa_loader
    if random.random() < MIX_RATIO:
        try:
            batch = next(medqa_loader)
        except StopIteration:
            medqa_loader = iter(make_loader(medqa_dataset, cfg.MED_SFT_BATCH_SIZE, pad_id))
            batch = next(medqa_loader)
    else:
        try:
            batch = next(pubmedqa_loader)
        except StopIteration:
            pubmedqa_loader = iter(make_loader(pubmedqa_dataset, cfg.MED_SFT_BATCH_SIZE, pad_id))
            batch = next(pubmedqa_loader)
    return {k: v.to(device) for k, v in batch.items()}

# Quick sanity check — inspect one batch
sample_batch = get_sft_batch()
print("Sample SFT batch:")
for k, v in sample_batch.items():
    print(f"  {k}: shape={v.shape}  dtype={v.dtype}")
    if k == "labels":
        n_ignored  = (v == -100).sum().item()
        n_active   = (v != -100).sum().item()
        print(f"    → {n_ignored} prompt tokens masked, {n_active} response tokens active")

## 5 — Load DAPT Checkpoint

We warm-start SFT from the DAPT checkpoint rather than from random weights or from the
raw TinyStories model.

Warm-starting from DAPT gives the model:
1. General language understanding (from TinyStories pretraining)
2. Medical vocabulary and domain knowledge (from DAPT)
3. Instruction following (learned during SFT)

Without DAPT, the SFT model would need to simultaneously learn both medical knowledge
and instruction format — a harder optimisation problem within only 5k steps.

In [ ]:
import torch
from model.gpt import GPT, GPTConfig
import config as cfg

MED_BLOCK_SIZE = 512

sft_config = GPTConfig(
    vocab_size=cfg.VOCAB_SIZE,
    block_size=MED_BLOCK_SIZE,
    n_layer=cfg.N_LAYERS,
    n_head=cfg.N_HEADS,
    n_embd=cfg.D_MODEL,
)

model = GPT(sft_config).to(device)

print(f"Loading DAPT weights from: {cfg.MED_DAPT_FINAL_CKPT}")
state = torch.load(cfg.MED_DAPT_FINAL_CKPT, map_location=device)
if isinstance(state, dict) and "model" in state:
    state = state["model"]
model.load_state_dict(state)

print(f"Model loaded — {model.num_parameters() / 1e6:.2f}M parameters")

# Save a copy of the DAPT model for the before/after comparison later
import copy
dapt_model = copy.deepcopy(model).to(device)
dapt_model.eval()
print("DAPT reference copy created for before/after comparison.")

## 6 — SFT Training Loop

**Training details:**
- 5,000 steps at `cfg.MED_SFT_BATCH_SIZE` examples per step
- Lower LR than DAPT (`cfg.MED_SFT_LR = 3e-5`) — we are fine-tuning, not adapting from scratch
- Loss is computed **only on response tokens** (prompt tokens have label=-100)
- Checkpoint every `cfg.MED_SFT_SAVE_INTERVAL` steps
- Eval loss every `cfg.MED_SFT_EVAL_INTERVAL` steps

**Expected loss:**
- Step 0: ~6.0 (model has never seen the instruction format)
- Step 1k: ~3.0–3.5 (instruction format learned)
- Step 5k: ~1.5–2.0 (model reliably selects the correct answer letter)

In [ ]:
import math
import torch
import config as cfg

def get_sft_lr(step):
    """Learning rate schedule for SFT: linear warmup then cosine decay.

    Uses a smaller peak LR than DAPT (cfg.MED_SFT_LR) because we are
    fine-tuning existing knowledge rather than learning from scratch.

    Args:
        step: current training step

    Returns:
        float: learning rate
    """
    warmup = 200   # short warmup for SFT
    total  = cfg.MED_SFT_MAX_STEPS
    peak   = cfg.MED_SFT_LR
    if step < warmup:
        return peak * (step + 1) / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return peak * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.MED_SFT_LR,
    weight_decay=0.01,   # lighter regularization for fine-tuning
    betas=(0.9, 0.95),
)

print(f"SFT optimizer ready. Peak LR: {cfg.MED_SFT_LR:.1e}")

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def estimate_sft_val_loss(n_batches=20):
    """Estimate SFT validation loss over n_batches random batches.

    Masks prompt tokens (label=-100) in the loss, same as training.

    Returns:
        float: mean cross-entropy loss on response tokens only
    """
    model.eval()
    losses = []
    for _ in range(n_batches):
        batch  = get_sft_batch()
        logits = model(batch["input_ids"])
        labels = batch["labels"]
        # Flatten to (B*T, vocab) for cross_entropy
        loss = F.cross_entropy(
            logits.view(-1, sft_config.vocab_size),
            labels.view(-1),
            ignore_index=-100,   # -100 = prompt tokens, ignored
        )
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

In [ ]:
# ── Resume logic ──────────────────────────────────────────────────────────
start_step = 0
if os.path.exists(cfg.MED_SFT_CKPT):
    print(f"Resuming SFT from: {cfg.MED_SFT_CKPT}")
    ckpt = torch.load(cfg.MED_SFT_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"]
    print(f"  → step {start_step:,}")
else:
    print("Starting SFT from step 0.")

In [ ]:
%%time
# ── SFT Training Loop ─────────────────────────────────────────────────────
model.train()
sft_loss_log = []

for step in range(start_step, cfg.MED_SFT_MAX_STEPS):

    lr = get_sft_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    batch  = get_sft_batch()
    logits = model(batch["input_ids"])
    labels = batch["labels"]

    # Loss is computed only on response tokens (prompt tokens masked with -100)
    loss = F.cross_entropy(
        logits.view(-1, sft_config.vocab_size),
        labels.view(-1),
        ignore_index=-100,
    )

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 50 == 0:
        print(f"step {step:>5} | lr {lr:.2e} | loss {loss.item():.4f}")

    if step % cfg.MED_SFT_EVAL_INTERVAL == 0 and step > 0:
        val_loss = estimate_sft_val_loss()
        sft_loss_log.append((step, loss.item(), val_loss))
        print(f"step {step:>5} | VAL loss {val_loss:.4f}")

    if step % cfg.MED_SFT_SAVE_INTERVAL == 0 and step > 0:
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step},
            cfg.MED_SFT_CKPT,
        )
        print(f"  Checkpoint → {cfg.MED_SFT_CKPT}")

torch.save(model.state_dict(), cfg.MED_SFT_FINAL_CKPT)
print(f"\nSFT complete. Model saved → {cfg.MED_SFT_FINAL_CKPT}")

## 7 — Before vs After: DAPT → SFT

We generate from both the DAPT model and the SFT model using the same 3 prompts.

**Expected difference:**
- **DAPT model**: completes the prompt as free-form medical prose — does not follow the MCQ format
- **SFT model**: selects an answer letter (A/B/C/D) and optionally provides an explanation —
  it has learned the instruction format

This comparison is the clearest demonstration that SFT teaches format, not just knowledge.

In [ ]:
from generation.sampler import generate, encode_prompt
from loaders.medical import format_medqa_instruction
from datasets import load_dataset
import textwrap
import config as cfg

med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)

# Use 3 examples from the MedQA test set (not seen during SFT)
raw_medqa = load_dataset(cfg.MED_MEDQA_DATASET, split="train")
# Use examples beyond the training range as a proxy test set
test_start = cfg.MED_MEDQA_TRAIN_SIZE
test_examples = list(raw_medqa.select(range(test_start, min(test_start + 3, len(raw_medqa)))))

GEN_KWARGS = dict(
    block_size=MED_BLOCK_SIZE,
    max_new_tokens=100,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.1,
)

model.eval()

for i, ex in enumerate(test_examples):
    fmt    = format_medqa_instruction(ex)
    prompt = fmt["prompt"]
    answer = fmt["response"]

    x = encode_prompt(prompt, med_tok, device)

    # DAPT generation
    dapt_out = generate(dapt_model, x, med_tok, **GEN_KWARGS)
    dapt_text = med_tok.decode(dapt_out[0].tolist())

    # SFT generation
    sft_out = generate(model, x, med_tok, **GEN_KWARGS)
    sft_text = med_tok.decode(sft_out[0].tolist())

    print(f"\n{'='*70}")
    print(f"Example {i+1}")
    print(f"PROMPT (truncated): {prompt[:300]}")
    print(f"\nGOLD ANSWER: {answer}")
    print(f"\n-- DAPT model --")
    print(textwrap.fill(dapt_text[:400], 80))
    print(f"\n-- SFT model --")
    print(textwrap.fill(sft_text[:400], 80))

model.train()

## Next Steps

The SFT model is saved at `cfg.MED_SFT_FINAL_CKPT`.

- **Notebook 14** — Evaluate MCQ accuracy and medical perplexity across all three checkpoints
  (TinyStories base, DAPT, SFT)
- **Notebook 15** — DPO: use MedQA's labeled wrong options as rejected responses to further
  align the model toward correct medical answers